# Space-Time Volume — SLA Resin & Casting Pipeline

```
engine/
  volume.py          ← SpaceTimeVolume 3D scalar field container
  video_ingest.py    ← Mode 1: OpenCV video ingestor
  optical_flow.py    ← Mode 1: Farneback inter-frame interpolation (anti-sawtooth)
  sdf_shapes.py      ← SDF primitive library (disc, rect, ring, capsule, smooth-union)
  procedural.py      ← Mode 2: SDF morphing via mathematical easing curves
  easing.py          ← 10 vectorised easing curves
  mesh_export.py     ← Chunked marching cubes + Negative Mold boolean subtraction + SLA tuning
  dummy_data.py      ← SDF synthetic keyframes & video arrays

viewer.py            ← Plotly FigureWidget + ipywidgets control panel
app.py               ← Composition root
```

**Run from project directory:**
```bash
pip install -r requirements.txt
jupyter lab demo.ipynb
```

---
## 1 · Launch the Full Interactive Viewer

Includes:
• Single flat `go.Surface` plane dynamically updated via Time (Z) slider
• Mode 1 (Farneback optical flow) ↔ Mode 2 (SDF Procedural with easing curves)
• **Negative Mold toggle** (Boolean subtraction for pouring wax or silicone)
• **SLA Tuning**: `step_size` (1 for max SLA resolution) & reduced Taubin smoothing passes
• **Memory Chunking slider** (`chunk_size=32`) to strictly obey 8GB System RAM limits

In [ ]:
from app import build_viewer

viewer = build_viewer(
    video_path=None,         # None → synthetic dummy; or r'C:\path\clip.mp4'
    sigma_t=0.8,             # final Gaussian polish (Mode 1)
    use_optical_flow=True,   # Farneback dense optical flow interpolation
    interp_factor=4,         # 4× time-axis upsampling
    n_proc_frames=48,        # frames per procedural transition
    edge_width=2.5,          # SDF gradient softness in pixels
    stl_step_size=1,         # 1 = max resolution for high-res SLA resin
    stl_smooth_iters=4,      # default 4 Taubin passes (preserves fine pixel details)
    chunk_size=32,           # chunked marching cubes for 8GB RAM
    stl_output_dir='output',
)

viewer.show()

---
## 2 · Engine API — Headless Pipeline

### 2a · Mode 1: Video Ingestion + Optical Flow

In [ ]:
from engine import VideoIngestor, OpticalFlowInterpolator, dummy_video_array

# Synthesize test video stack
raw_arr = dummy_video_array(n=12, h=64, w=96)
raw_vol = VideoIngestor.from_array(raw_arr)
print(f"Raw volume: {raw_vol}")

# 4× sub-frame interpolation via Farneback optical flow
ofi = OpticalFlowInterpolator(interp_factor=4, winsize=15)
flow_vol = ofi.interpolate(raw_vol)
print(f"Optical flow interpolated: {flow_vol} (Z: {raw_vol.n_frames} → {flow_vol.n_frames} frames)")

# Light anisotropic Gaussian polish
polished_vol = flow_vol.smoothed(sigma_t=0.8)
print(f"Polished volume: {polished_vol}")

### 2b · Mode 2: SDF Procedural Morphing

In [ ]:
from engine import ProceduralGenerator, dummy_keyframes

frame_a, frame_b = dummy_keyframes(h=64, w=96)
gen = ProceduralGenerator(frame_a, frame_b, edge_width=2.5)
print(f"Centroid displacement: Δy={gen.displacement[0]:.2f}px  Δx={gen.displacement[1]:.2f}px")

# Smoothstep easing in SDF coordinate space
proc_vol = gen.generate(32, 'smoothstep (3t²−2t³)')
print(f"Procedural volume: {proc_vol}")
print(f"Morphing strategy: {proc_vol.meta['morphing']}")

---
## 3 · Hardware Export: High-Res SLA Mesh & Negative Mold Generation

Compare exporting:
1. **Positive Model** (tuned for SLA resin with `step_size=1` and `smooth_iterations=4`)
2. **Negative Mold** (solid 3D bounding box minus Space-Time Volume, with open pour spout for pouring wax or silicone)

In [ ]:
from pathlib import Path
from engine import ProceduralGenerator, dummy_keyframes, MeshExporter

Path('output').mkdir(exist_ok=True)
fa, fb = dummy_keyframes(h=48, w=64)
vol = ProceduralGenerator(fa, fb, edge_width=2.5).generate(28, 'ease-in-out cubic')

# 1. Positive Model (High-Res SLA Resin)
exp_sla = MeshExporter(
    level=0.5,
    step_size=1,          # Maximum SLA resolution
    smooth_iterations=4,  # Preserves sharp pixel details for resin
    voxel_size=0.5,
    chunk_size=16,        # Chunked processing for 8GB RAM
)
stats_pos = exp_sla.export(vol, 'output/sla_positive_model.stl')
print("=== Positive Model (SLA) ===")
print(f"  Verts: {stats_pos.n_vertices:,} | Faces: {stats_pos.n_faces:,} | Watertight: {stats_pos.is_watertight}")
print(f"  Volume: {stats_pos.volume_mm3:.1f} mm³ | Path: {stats_pos.output_path}")

# 2. Negative Mold (Boolean Subtraction for Wax/Silicone Casting)
exp_mold = MeshExporter(
    level=0.5,
    step_size=1,
    smooth_iterations=4,
    voxel_size=0.5,
    chunk_size=16,
    as_negative_mold=True,
    mold_wall_thickness=4, # 4-voxel solid bounding box wall
    mold_open_top=True,    # Open pour hole at the top
)
stats_mold = exp_mold.export(vol, 'output/silicone_negative_mold.stl')
print("\n=== Negative Mold (Casting) ===")
print(f"  Verts: {stats_mold.n_vertices:,} | Faces: {stats_mold.n_faces:,} | Watertight: {stats_mold.is_watertight}")
print(f"  Volume: {stats_mold.volume_mm3:.1f} mm³ | Path: {stats_mold.output_path}")

---
## 4 · 3D Mesh Inspection: Positive vs. Negative Mold

Render the exported meshes side-by-side using Plotly `Mesh3d`.

In [ ]:
import trimesh
import plotly.graph_objects as go
import numpy as np

mesh_pos  = trimesh.load('output/sla_positive_model.stl')
mesh_mold = trimesh.load('output/silicone_negative_mold.stl')

vp, fp = np.array(mesh_pos.vertices), np.array(mesh_pos.faces)
vm, fm = np.array(mesh_mold.vertices), np.array(mesh_mold.faces)

fig = go.Figure()

# Positive Model
fig.add_trace(go.Mesh3d(
    x=vp[:, 0], y=vp[:, 1], z=vp[:, 2],
    i=fp[:, 0], j=fp[:, 1], k=fp[:, 2],
    name='Positive SLA Model',
    intensity=vp[:, 2], colorscale='Viridis',
    flatshading=False,
    lighting=dict(ambient=0.4, diffuse=0.8, specular=0.5),
))

# Negative Mold (shifted along X axis for side-by-side inspection)
x_offset = float(mesh_pos.bounds[1, 0] - mesh_pos.bounds[0, 0]) * 1.8
fig.add_trace(go.Mesh3d(
    x=vm[:, 0] + x_offset, y=vm[:, 1], z=vm[:, 2],
    i=fm[:, 0], j=fm[:, 1], k=fm[:, 2],
    name='Negative Mold (Open Pour Top)',
    intensity=vm[:, 2], colorscale='Plasma',
    flatshading=False,
    lighting=dict(ambient=0.4, diffuse=0.8, specular=0.5),
))

fig.update_layout(
    height=620,
    title='Side-by-Side: Positive SLA Model vs. Negative Silicone Mold',
    scene=dict(aspectmode='data', bgcolor='rgb(12,14,20)'),
    paper_bgcolor='rgb(18,20,28)',
    font=dict(color='#ddd'),
)
fig.show()

---
## 5 · Strict Memory Management Verification (8GB RAM)

Verifying that `chunked_marching_cubes` executes with flat memory usage and $O(1)$ empty-chunk skipping via SciPy sparse slice indexing.

In [ ]:
from engine import chunked_marching_cubes, SparseOccupancyIndex
import numpy as np

# Create dense volumetric scalar field
dense_vol = np.zeros((80, 100, 100), dtype=np.float32)
dense_vol[20:60, 25:75, 25:75] = 0.8  # only middle 40 slices active

occ = SparseOccupancyIndex(dense_vol, level=0.5)
print(f"Total slices: {occ.total_slices} | Active slices with isosurface: {occ.sparse_active.nnz}")

# Run chunked marching cubes
verts, faces = chunked_marching_cubes(dense_vol, level=0.5, step_size=1, chunk_size=16)
print(f"Extracted mesh: {len(verts):,} vertices, {len(faces):,} faces (Strict memory bound OK!)")